# Tabular Deep Learning & Classification Pipeline with FastAI

## Overview
This notebook implements an end-to-end deep learning pipeline for structured tabular data using the UCI Adult Census Income dataset. It demonstrates automated feature preprocessing, embedding creation for categorical variables, continuous variable normalization, and classification training using the 1-Cycle policy.

## Key Technical Components
* **Dataset & Feature Engineering:** Uses FastAI's `TabularDataLoaders.from_csv` to ingest tabular records, distinguishing between discrete categorical features (`workclass`, `education`, `marital-status`, `occupation`, `relationship`, `race`) and continuous numeric features (`age`, `fnlwgt`, `education-num`).
* **Preprocessing Transforms (`procs`):**
  * `Categorify`: Encodes high-cardinality categorical strings into categorical embedding-ready integer IDs.
  * `FillMissing`: Automatically detects and handles missing continuous values by computing the median and generating a companion boolean missing indicator column.
  * `Normalize`: Performs z-score feature scaling across continuous numerical variables.
* **Architecture & Training:** Instantiates a multi-layer perceptron (MLP) with categorical embeddings via `tabular_learner`, optimized using the `fit_one_cycle` scheduling policy.
* **Evaluation Metrics:** Evaluates binary classification performance using balanced metrics: overall `accuracy` alongside `F1Score` (harmonic mean of precision and recall) to account for potential class imbalance.

---------
---------
---------

# Tabular Analysis - Income Prediction

This is the most widely used model used in industry. Here datasource isn't folder of JPEGs/PNGs instead a .csv or a dataframe.

In [1]:
!pip install -Uqq "numpy>=2.0.0,<2.1.0" "fastai>=2.7.15"

# "Goldilocks" version of Numpy

In [2]:
from fastai.tabular.all import *

# vision.all imports ResNet, ImageDataLoaders, aug_transforms while 
# tabular.all imports tabular_learner, procs like Categorify and FillMissing, optimization logic like fit_one_cycle

In [3]:
# using 'salary' in both y_names and cat_names will create "Double-dip" and display multiple salary columns in show_batch as 
#  'salary' is target and we are providing the model with the answer key so nothing to guess (100% accuracy)
# education-num_na isn't also required as FillMissing creates that column automatically and if we do that fast_ai gets confused

# fnlwgt is the final weight. its an estimate if no. of individuals in target population that the record represents.

In [4]:
path = untar_data(URLs.ADULT_SAMPLE) # Compressed dataset is downloaded and returns the local folder path.
# "Adult" dataset is a classic. It contains census data used to predict if someone earns more than $50k/year.

# The Dataloaders(The Engine Room)
dls = TabularDataLoaders.from_csv(path/'adult.csv', path = path, y_names = "salary", 
    cat_names = ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race'], #categorical
    cont_names = ['age', 'fnlwgt', 'education-num'], #continuous
    procs = [Categorify, FillMissing, Normalize])

# path/'adult.csv' is the source file where data is originally present after untar_data
# path = path is the Base(Home) directory where updated(after operations done) model is saved. when we eventually run 'learn.save('my_model')'
#  it creates a new subfolder 'models/'' under path folder and saves a file 'my_model/pth' inside that subfolder.
'''
    y_names = 'salary' tells fastai which is the "Target" column (which we want to predict)
    cat_names is the category names (things with labels like {workclass:'Private','Self-employed'})
    cont_names is the continuous numbers (things with scale like {Age: 20 to 60})
'''

'''
  The Preprocessors: procs = [Categorify, FillMissing, Normalize]
  In CV projects, we use "Transforms(item_tfms)" like 'Resize', 'aug_transforms' but in Tabular we use 'Procs'
1. Categorify: Replaces the string "Private" with a number like 1
2. FillMissing: Finds empty cells. It fills them with the median and adds a new column ending in _na to tell the model, "I guessed this value".
    FillMissing only touches continuous variables like Age(Age_na created) not categorical values.
    Categorical values are dealt with Categorify where it treats them like a separate category(It reserves 0 for "missing" or "other" and other categorical values as 1,2,3 etc.).
3. Normalize: It subtracts the mean and divides by the standard deviation. It ensures a high "Age" doesn't drown out a small "Education-num."
    Used to normalize magnitudes like a change of 1 in age (20 to 21 is 5% jump) but 1 in income (100,000 to 100,001 is 0.001% jump).
    Without normalization giant numbers often overwhelm small numbers. Here education-num is 1-16 which may look like "noise" compate to huge swings in other columns.
'''

/usr/local/lib/python3.12/dist-packages/fastai/tabular/core.py:314: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  to[n].fillna(self.na_dict[n], inplace=True)


'\n  The Preprocessors: procs = [Categorify, FillMissing, Normalize]\n  In CV projects, we use "Transforms(item_tfms)" like \'Resize\', \'aug_transforms\' but in Tabular we use \'Procs\'\n1. Categorify: Replaces the string "Private" with a number like 1\n2. FillMissing: Finds empty cells. It fills them with the median and adds a new column ending in _na to tell the model, "I guessed this value".\n    FillMissing only touches continuous variables like Age(Age_na created) not categorical values.\n    Categorical values are dealt with Categorify where it treats them like a separate category(It reserves 0 for "missing" or "other" and other categorical values as 1,2,3 etc.).\n3. Normalize: It subtracts the mean and divides by the standard deviation. It ensures a high "Age" doesn\'t drown out a small "Education-num."\n    Used to normalize magnitudes like a change of 1 in age (20 to 21 is 5% jump) but 1 in income (100,000 to 100,001 is 0.001% jump).\n    Without normalization giant numbers o

In [5]:
print(path) # This is the temporary virtual machine path assigned to runnning session.
path.ls()

/root/.fastai/data/adult_sample


(#3) [Path('/root/.fastai/data/adult_sample/models'),Path('/root/.fastai/data/adult_sample/adult.csv'),Path('/root/.fastai/data/adult_sample/export.pkl')]

In [6]:
dls.show_batch()
# It visualize the whole data. It shows a dataframe snippet. It's checking if the columns look right and the labels are correctly mapped.

,workclass,education,marital-status,occupation,relationship,race,education-num_na,age,fnlwgt,education-num,salary
0,Local-gov,11th,Divorced,Adm-clerical,Unmarried,White,False,39.000000,189911.000002,7.0,<50k
1,Private,Some-college,Never-married,Other-service,Own-child,White,False,20.000000,332931.003355,10.0,<50k
2,Self-emp-not-inc,Bachelors,Married-civ-spouse,Craft-repair,Husband,White,False,49.000000,126267.998035,13.0,<50k
3,Private,Bachelors,Never-married,Adm-clerical,Own-child,White,False,24.000001,72118.997422,13.0,<50k
4,Private,HS-grad,Married-civ-spouse,Craft-repair,Husband,White,False,41.000000,49796.995913,9.0,<50k
5,Private,HS-grad,Divorced,Machine-op-inspct,Unmarried,Black,True,37.000000,175390.000766,10.0,<50k
6,Private,Bachelors,Divorced,Sales,Unmarried,White,False,46.000000,228371.998606,13.0,>=50k
7,Local-gov,12th,Married-civ-spouse,Transport-moving,Husband,White,False,43.000000,301637.994297,8.0,<50k
8,Self-emp-not-inc,Bachelors,Never-married,Craft-repair,Own-child,White,False,36.000000,206520.000760,13.0,<50k
9,Private,Some-college,Married-civ-spouse,Sales,Husband,White,False,41.000000,104333.999644,10.0,>=50k


In [7]:
'''
fit_one_cycle method is based on "One Cycle Policy". Unlike traditional training where the learning rate stays the same or goes down, 
this policy does something counter-intuitive: it speeds up before it slows down.
Mechanism: (Ball rolling down hill example)
 1. Phase 1(The Warm-up): The learning rate starts very low and increases for the first half of the cycle. This "momentum" helps the 
     ball roll out of small, shallow dips(local minima) that aren't the best solution.
 2. Phase 2(The Cool-down): The learning rate decreases for the second half. This allows the ball to stop "bouncing" and settle precisely
     into the bottom of the deepest valley it found.
'''

'''
2,3 are the no. of epochs or the no. of times this warm-up and cool-down cycle happening. Generally 1-2 okay , 3+ generally causes overfitting.
'''

'''
EPOCH TABLE
Scenario        Train Loss      Val Loss       Meaning

Learning        Decreasing      Decreasing     Perfect. The model is finding general patterns. Keep going!
Overfitting     Decreasing      Increasing     "Bad. The model is memorizing the training data. It’s ""cheating"" and won't work on new data."
Underfitting    High/Flat       High/Flat      "Stuck. Your learning rate might be too low, or you need more epochs."
"The Sweetspot" Decreasing      Flat (at min)  Stop here. This is the lowest the Validation Loss will go before it starts rising again.
'''

'\nEPOCH TABLE\nScenario        Train Loss      Val Loss       Meaning\n\nLearning        Decreasing      Decreasing     Perfect. The model is finding general patterns. Keep going!\nOverfitting     Decreasing      Increasing     "Bad. The model is memorizing the training data. It’s ""cheating"" and won\'t work on new data."\nUnderfitting    High/Flat       High/Flat      "Stuck. Your learning rate might be too low, or you need more epochs."\n"The Sweetspot" Decreasing      Flat (at min)  Stop here. This is the lowest the Validation Loss will go before it starts rising again.\n'

In [11]:
learn = tabular_learner(dls, metrics = [accuracy, F1Score()]) #F1-score is harmonic mean of precision and accuracy
learn.fit_one_cycle(2)

# As accuracy is the mean it can be wrong if 90% of people in dataset earn <50k so model guess <50k for everyone. Other parameters to use 
# are Precision, Recall, F1 Score, ROC AUC.

# tabular_learner looks at 'dls' and builds a custom brain from stratch specifically for our spreadsheet's width and column types. It 
#  creates "Embedding layers" for our categories and "Linear Layers for our numbers"

# learn.fit_one_cycle(): This is Leslie N. Smith


# fit_one_cycle and not finetune used in tabular model as there is no pre-trained model
# In vision, we use a NN that already "know" what edges and shapes look like. In Tabular, every dataset is unique(one spreadsheet's "Age"
#  column has nothing to do with another's "Price" column), so we always start from stratch(random weights).

epoch,train_loss,valid_loss,accuracy,f1_score,time
0,0.372187,0.360679,0.835227,0.641497,00:02
1,0.351716,0.346039,0.845670,0.651405,00:02


In [12]:
learn.show_results() # It shows table of Actual salary vs Predicted values by model.

,workclass,education,marital-status,occupation,relationship,race,education-num_na,age,fnlwgt,education-num,salary,salary_pred
0,5.0,12.0,3.0,2.0,6.0,5.0,1.0,-0.923462,-1.497156,-0.425747,1.0,0.0
1,8.0,16.0,5.0,14.0,2.0,5.0,1.0,-0.410436,0.509524,-0.033393,0.0,0.0
2,5.0,10.0,3.0,13.0,1.0,5.0,1.0,0.322457,-0.607411,1.143671,1.0,1.0
3,5.0,9.0,5.0,4.0,2.0,5.0,1.0,-1.216619,-0.898406,0.358962,0.0,0.0
4,5.0,3.0,5.0,7.0,4.0,1.0,1.0,-1.436487,-1.511253,-0.818102,0.0,0.0
5,1.0,16.0,7.0,1.0,5.0,5.0,1.0,1.861534,-0.803036,-0.033393,0.0,0.0
6,5.0,16.0,5.0,8.0,2.0,5.0,1.0,-0.557015,-0.965743,-0.033393,0.0,0.0
7,5.0,12.0,3.0,4.0,1.0,5.0,1.0,-0.117279,0.879836,-0.425747,0.0,0.0
8,5.0,12.0,7.0,2.0,5.0,5.0,1.0,-0.337147,-1.561035,-0.425747,0.0,0.0
